### Necessary Imports

In [ ]:
import os
from chembl_webresource_client.new_client import new_client
import pandas as pd
import json
import urllib.request, urllib.parse, urllib.error
import requests
import re
import time
from rdkit import Chem
from rdkit.Chem import inchi, PandasTools, MolStandardize
from chembl_structure_pipeline import standardizer

### Data Retrieval Function
 - `retrieve_chembl_data()`: Gathers bioactivity data from ChEMBL for list of provided chembl_ids. Outputs pandas dataframe with collected data.

In [2]:
def retrieve_chembl_data(chembl_ids):
    total = []

    for id in chembl_ids:
        bioactivities = new_client.activity.filter(target_chembl_id=id)
        bioactivities = pd.DataFrame(bioactivities)
        total.append(bioactivities)
        print(f'Number of records for {id} is', bioactivities.shape[0])

    full_df = pd.concat(total)

    return full_df

### Retrieve ChEMBL Data
 - Retrieve P-gp, BCRP, and MRP1 data from CHEMBL_IDs
 - P-gp: CHEMBL4302
 - BCRP: CHEMBL5393
 - MRP1: CHEMBL3004

In [3]:
chembl_ids = ['CHEMBL4302','CHEMBL5393','CHEMBL3004']

In [4]:
# chembl_df = retrieve_chembl_data(chembl_ids)
# chembl_df.to_csv('abc_transporters_2025_08_27.csv')

### Standard SMILES Retrieval Functions
 - `fetch_with_retries()`: helper function for standard_smiles(). Addresses temporary errors that may arise when attempting PubChem data retrieval.
 - `inchikey_fallback()`: helper function for standard_smiles(). Search PubChem using compounds InChI key if SMILES search fails.
 - `standard_smiles()`: 

In [5]:
def fetch_with_retries(url, retries=3, delay=1):
    """Attempt to gather SMILES from PubChem with up to `retries` attempts. Increases delay if rate-limited or if other temporary error occurs."""
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                return response
            elif response.status_code == 429:
                time.sleep(delay)
                delay *= 2
            else:
                return None

        except requests.exceptions.RequestException:
            if attempt == retries - 1:
                print(f"Request exception for URL: {url}")
            time.sleep(delay)
            delay *= 2

    print(f"Failed after {retries} retries: {url}")
    return None


def inchikey_fallback(smiles: str) -> str:
    """Convert a SMILES string to InChIKey and generate PubChem lookup URL."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    ikey = inchi.MolToInchiKey(mol)
    return f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{ikey}/property/ConnectivitySMILES/JSON"

def standard_smiles(df, delay=0.5):
    df = df.copy()
    df.columns = df.columns.str.strip()

    if 'molecule_chembl_id' not in df.columns or 'canonical_smiles' not in df.columns:
        raise KeyError(f"Missing required columns: 'molecule_chembl_id' or 'canonical_smiles'")
    id_smiles_map = (
        df.dropna(subset=['molecule_chembl_id', 'canonical_smiles'])
          .drop_duplicates('molecule_chembl_id').set_index('molecule_chembl_id')['canonical_smiles'].to_dict())

    total = len(id_smiles_map)
    print(f"[INFO] {total} unique compounds for {len(df)} entries: Retrieving SMILES...")
    
    chembl_to_smiles = {}

    progress_display = display("", display_id=True)
    
    for i, (chembl_id, smiles_str) in enumerate(id_smiles_map.items(), start=1):
        smiles = None
        url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{chembl_id}/property/ConnectivitySMILES/JSON"
        response = fetch_with_retries(url)

        if response and response.status_code == 200:
            data = response.json()
            smiles = data['PropertyTable']['Properties'][0].get('ConnectivitySMILES')
        else:
            url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{smiles_str}/property/ConnectivitySMILES/JSON"
            response = fetch_with_retries(url)

            if response and response.status_code == 200:
                data = response.json()
                smiles = data['PropertyTable']['Properties'][0].get('ConnectivitySMILES')
            else:
                inchikey_url = inchikey_fallback(smiles_str)
                if inchikey_url:
                    response = fetch_with_retries(inchikey_url)
                    if response and response.status_code == 200:
                        data = response.json()
                        smiles = data['PropertyTable']['Properties'][0].get('ConnectivitySMILES')

        if smiles is None:
            print(f"[INFO] Removing unresolved compound: {chembl_id}")

        chembl_to_smiles[chembl_id] = smiles
        progress_display.update(f"[PROGRESS] Processed {i} / {total} compounds")
        time.sleep(delay)

    df['SMILES'] = df['molecule_chembl_id'].map(chembl_to_smiles)
    df = df[df['SMILES'].notna()]

    print(f"[INFO] {len(df)} entries retained for {df['molecule_chembl_id'].nunique()} unique compounds.")
    return df

### Molecular Representation Standardization Functions
 - `standardize_mol()`: Standardizes a given molecule using ChEMBL curation pipeline standardizer.
 - `process_mols()`: deploys standardize_mol() on entire SDF file. Output: standardized SDF file ("{filename}_CSP.SDF").

In [6]:
def standardize_mol(molecule):
    """
    Standardizes a molecule using the standardizer from the ChEMBL curation pipeline
    :param molecule: RDKit molecule object (type: obj)

    :return: standardized RDKit molecule (type: obj)
    """
    desalter = MolStandardize.rdMolStandardize.LargestFragmentChooser()
    std1_mol = standardizer.standardize_mol(molecule)
    desalt_mol = desalter.choose(std1_mol)
    std2_mol = standardizer.standardize_mol(desalt_mol)
    return std2_mol

def process_mols(input_file, mol_column='ROMol'):
    """
    Processes/standardizes multiple molecules from an SD file and saves them to a new SD file

    :param input_file: SD file to be processed (type: str)
    :param mol_column: the name of the molecule column in the SD file (default: 'ROMol') (type: str)

    :return: DataFrame with the new standardized molecules
    """

    basename = os.path.basename(input_file)
    directory = os.path.dirname(input_file)
    filename = os.path.splitext(basename)[0]

    sdf = Chem.PandasTools.LoadSDF(input_file)
    molecules = [x for x in sdf[mol_column]]
    new_molecules = []

    for i in range(len(molecules)):
        new_mol = standardize_mol(molecules[i])
        new_molecules.append(new_mol)

    sdf_new = sdf.copy()
    sdf_new[mol_column] = new_molecules

    filepath = os.path.join(directory, f'{filename}_CSP.sdf')
    PandasTools.WriteSDF(sdf_new, filepath, properties=list(sdf_new.columns))

    return sdf_new

### Transporter-Specific Dataset Curation
 1. Load ChEMBL dataset
 2. Assign dictionary map for transporter CHEMBL_IDs and names
 3. Curate data based on standard_type (IC50), pchembl_value, SMILES, standard_relation (=), and data_validity_comments
 4. Standardize molecular SMILES
 5. Save SDF for each transporter
 6. Process datasets: Clean and standardize molecules

In [7]:
chembl_df = pd.read_csv('abc_transporters_2025_08_27.csv', low_memory=False)

In [8]:
abc_dict = {'CHEMBL4302':'pgp','CHEMBL5393':'bcrp','CHEMBL3004':'mrp1'}

In [ ]:
abc_dfs = {}
for c,t in abc_dict.items():
    abc_dfs[t] = chembl_df[chembl_df.target_chembl_id == c]
    abc_dfs[t] = abc_dfs[t][abc_dfs[t].standard_type == 'IC50']
    abc_dfs[t] = abc_dfs[t].dropna(subset='pchembl_value')
    abc_dfs[t] = abc_dfs[t].dropna(subset='canonical_smiles')
    abc_dfs[t] = abc_dfs[t][abc_dfs[t].standard_relation == '=']
    print(abc_dfs[t].data_validity_comment.isna().value_counts(), '\n')
    abc_dfs[t] = abc_dfs[t][['molecule_chembl_id','canonical_smiles','pchembl_value',]]
    abc_dfs[t] = standard_smiles(abc_dfs[t],delay=0.3)
    abc_dfs[t] = abc_dfs[t].rename(columns={'molecule_chembl_id':'Compound_ID','pchembl_value':'ACTIVITY'})
    PandasTools.AddMoleculeColumnToFrame(abc_dfs[t],smilesCol='SMILES')
    display(c,t,abc_dfs[t][['Compound_ID','SMILES','ACTIVITY','ROMol']])
    sdf_filename = f"{t}_inhibition.sdf"
    PandasTools.WriteSDF(abc_dfs[t], sdf_filename, properties = abc_dfs[t].columns.tolist())
    process_mols(sdf_filename)

### Duplicate Handling and Final Check
 1. CaseUltra was used to average the activities of duplicate entries and ensure no duplicate or invalid entries remain.
 2. Transporter modeling sets (e.g., pgp_pic50_modeling_set_20250807_SC_CSP.sdf) were then ready for model development.